In [1]:
import os

import librosa
import numpy as np
import pandas as pd

# For tracking progress
from tqdm import tqdm

In [2]:
# Dynamically access klink root and subsequent paths
from pathlib import Path

project_root = Path.cwd().resolve().parent
csv_path = project_root / "data" / "processed"
raw_path = project_root / "data" / "raw"

In [3]:
def extract_audio_features(file_path, task='mood', sr=22050):
    valid_tasks = {'mood', 'genre', 'instrument'}
    if task not in valid_tasks:
        raise ValueError(f"Unsupported task '{task}'. Expected one of: {sorted(valid_tasks)}.")

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Audio file not found: {file_path}")

    try:
        # Load audio file
        y, sr = librosa.load(file_path, sr=sr)

        # Extract features
        features = {
            # MFCC (Mel-frequency cepstral coefficients) - Captures timbre
            "mfcc_std": np.std(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13 if task == 'mood' else 20), axis=1),
            "mfcc_mean": np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13 if task == 'mood' else 20), axis=1),

            # Spectral features; perceived brightness of sounds
            "spectral_centroid": np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)),
            "spectral_bandwidth": np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr)),
            "spectral_contrast": np.mean(librosa.feature.spectral_contrast(y=y, sr=sr), axis=1),
            "spectral_rolloff": np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)),

            # Chroma features - harmonic patterns and pitch content
            "chroma_stft_mean": np.mean(librosa.feature.chroma_stft(y=y, sr=sr)),

            # RMS
            "rms_mean": np.mean(librosa.feature.rms(y=y)),

            # Zero Crossing Rate - Measures noisiness
            "zcr": np.mean(librosa.feature.zero_crossing_rate(y=y)),

            # Mel-Spectrogram
            "mel_spec_mean": np.mean(librosa.feature.melspectrogram(y=y, sr=sr), axis=1),
        }

        if task == 'instrument':
            # Harmonic-to-Noise Ratio (HNR)
            features['hnr_mean'] = np.mean(librosa.effects.harmonic(y) / (librosa.effects.percussive(y) + 1e-6))
            # Onset Strength - Intensity of note onsets (sharp changes in energy)
            features['onset_strength'] = np.mean(librosa.onset.onset_strength(y=y, sr=sr))

        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        if task in {'genre', 'mood'}:
            # Tempo - Beats Per Minute
            features['tempo'] = tempo

        if task == 'mood':
            # Dynamics - variability in loudness
            features['rms_std'] = np.std(librosa.feature.rms(y=y))

        return features  # Return dictionary of extracted features

    except (FileNotFoundError, OSError, ValueError, TypeError, RuntimeError) as exc:
        print(f"Error processing {file_path}: {exc}")
        return None

In [ ]:
# Process EmoMusic dataset, extracting features and linking them to mood labels.
def process_filtered_emomusic(audio_dir, song_info_file):

    # Load song info (only annotated files)
    song_info = pd.read_csv(song_info_file)
    data = []  # To store features and labels

    for _, row in tqdm(song_info.iterrows(), total=len(song_info), desc="Processing EmoMusic"):
        song_id = str(row['song_id'])
        file_name = row['file_name']
        start_of_the_segment = row['start of the segment (min.sec)']
        end_of_the_segment = row['end of the segment (min.sec)']
        Genre = row['Genre']
        Mediaeval_2013_Set = row['Mediaeval 2013 set']
        mean_arousal = row['mean_arousal']
        std_arousal = row['std_arousal']
        mean_valence = row['mean_valence']
        std_valence = row['std_valence']

        # Locate audio file by song ID
        file_path = os.path.join(audio_dir, f"{song_id}.mp3")
        if os.path.exists(file_path):
            features = extract_audio_features(file_path, 'mood')
            if features:
                # Add features to data
                data.append({
                    "song_id": song_id,
                    "file_name": file_name,
                    "start_of_the_segment": start_of_the_segment,
                    "end_of_the_segment": end_of_the_segment,
                    "Genre": Genre,
                    "Mediaeval_2013_set": Mediaeval_2013_Set,
                    "mean_arousal": mean_arousal,
                    "std_arousal": std_arousal,
                    "mean_valence": mean_valence,
                    'std_valence': std_valence,
                    **features
                })
        else:
            print(f"Audio file not found for song ID: {song_id}")

    # Flatten feature dictionaries
    processed_data = []
    for feat in data:
        flat_feat = {key: feat[key] for key in ["song_id", "file_name", "start_of_the_segment", "end_of_the_segment", "Genre", "Mediaeval_2013_set", "mean_arousal", "std_arousal", "mean_valence", "std_valence", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "chroma_stft_mean", "rms_mean", "rms_std", "zcr", "tempo"]}  # Keep unflattened
        flat_feat.update({f'spectral_contrast_{i}': feat['spectral_contrast'][i] for i in range(len(feat['spectral_contrast']))})
        flat_feat.update({f'mfcc_mean_{i}': feat['mfcc_mean'][i] for i in range(len(feat['mfcc_mean']))})
        flat_feat.update({f'mfcc_std_{i}': feat['mfcc_std'][i] for i in range(len(feat['mfcc_std']))})
        flat_feat.update({f'mel_spec_mean_{i}': feat['mel_spec_mean'][i] for i in range(len(feat['mel_spec_mean']))})
        processed_data.append(flat_feat)

    return pd.DataFrame(processed_data)


In [ ]:
# Example usage
audio_dir = raw_path / "EMOMUSIC - clips_45seconds" / "Audio"
song_info_file = csv_path / "EMOMUSIC_merged_info.csv"

# Process dataset
audio_features_df = process_filtered_emomusic(audio_dir, song_info_file)

unflattened_columns = ["song_id", "file_name", "start_of_the_segment", "end_of_the_segment", "Genre", "Mediaeval_2013_set", "mean_arousal", "std_arousal", "mean_valence", "std_valence", "chroma_stft_mean", "rms_mean", "rms_std", "zcr", "tempo", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff"]
flattened_columns = [col for col in audio_features_df.columns if col not in unflattened_columns]

# Unflattened + flattened columns
audio_features_df = audio_features_df[unflattened_columns + flattened_columns]

# Save to CSV
audio_features_df.to_csv(csv_path / "EMOMUSIC_features_45_sec.csv", index=False)
print("Saved EmoMusic features to", csv_path / "EMOMUSIC_features_45_sec.csv")

Processing EmoMusic:   0%|          | 0/744 [00:00<?, ?it/s]

Processing EmoMusic:   1%|          | 5/744 [00:00<00:55, 13.20it/s]

Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\2.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\3.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\4.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\5.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\7.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\8.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodePr

Processing EmoMusic:   2%|▏         | 14/744 [00:00<00:27, 26.27it/s]

Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\13.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\17.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\18.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\19.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\20.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\21.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\Vs

...

Processing EmoMusic:  99%|█████████▉| 739/744 [00:22<00:00, 32.45it/s]

Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\990.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\991.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\993.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\994.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\995.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\996.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\wi

Processing EmoMusic: 100%|██████████| 744/744 [00:23<00:00, 32.23it/s]


Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\999.mp3: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\EMOMUSIC - clips_45seconds\Audio\1000.mp3: cannot load library 'libsndfile.dll': error 0x7e


KeyError: "None of [Index(['song_id', 'file_name', 'start_of_the_segment', 'end_of_the_segment',\n       'Genre', 'Mediaeval_2013_set', 'mean_arousal', 'std_arousal',\n       'mean_valence', 'std_valence', 'chroma_stft_mean', 'rms_mean',\n       'rms_std', 'zcr', 'tempo', 'spectral_centroid', 'spectral_bandwidth',\n       'spectral_rolloff'],\n      dtype='str')] are in the [columns]"

In [ ]:
#Process dataset, extracting features for dataframe
def process_dataset(dataset_path):
    data = [] #stores features and labels
    for label in os.listdir(dataset_path):
        label_path = os.path.join(dataset_path, label)
        if os.path.isdir(label_path): #Check if folder
            for file in tqdm(os.listdir(label_path), desc=f"Processing {label}"):
                file_path = os.path.join(label_path, file)
                if file_path.endswith(".wav"): #Process only .wav files
                    file_name, _ = os.path.splitext(os.path.basename(file_path))
                    features = extract_audio_features(file_path)
                    if features:
                        #Add filename, label, and features to dataset
                        data.append({"file_name": file_name, "label": label, **features})

    # Flatten feature dictionaries
    processed_data = []
    for feat in data:
        flat_feat = {key: feat[key] for key in ["file_name", "label", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "chroma_stft_mean", "rms_mean", "zcr", "hnr_mean", "onset_strength"]}  # Keep unflattened
        flat_feat.update({f'spectral_contrast_{i}': feat['spectral_contrast'][i] for i in range(len(feat['spectral_contrast']))})
        flat_feat.update({f'mfcc_mean_{i}': feat['mfcc_mean'][i] for i in range(len(feat['mfcc_mean']))})
        flat_feat.update({f'mfcc_std_{i}': feat['mfcc_std'][i] for i in range(len(feat['mfcc_std']))})
        flat_feat.update({f'mel_spec_mean_{i}': feat['mel_spec_mean'][i] for i in range(len(feat['mel_spec_mean']))})
        processed_data.append(flat_feat)

    #Convert to Pandas Dataframe
    return pd.DataFrame(processed_data)

In [7]:
#Path to my unzipped dataset
dataset_path = raw_path / "IRMAS-TrainingData"

#Process dataset
audio_features_df = process_dataset(dataset_path)

#Order Columns
unflattened_columns = ["file_name", "label", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "chroma_stft_mean", "rms_mean", "zcr", "hnr_mean", "onset_strength"]
flattened_columns = [col for col in audio_features_df.columns if col not in unflattened_columns]

#unflattened + flattened columns
audio_features_df = audio_features_df[unflattened_columns + flattened_columns]

#Save features to a csv for future use
audio_features_df.to_csv(csv_path / "IRMAS_features_2_sec.csv", index=False)

print("Feature extraction complete! Data saved to", csv_path / "IRMAS_features_2_sec.csv", ".")

Processing acoustic guitar:   1%|▏         | 8/637 [00:00<00:17, 36.87it/s]

Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\014__[gac][nod][cou_fol]0770__1.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\014__[gac][nod][cou_fol]0770__2.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\014__[gac][nod][cou_fol]0770__3.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\016__[gac][dru][cou_fol]0671__1.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\016__[gac][dru][cou_fol]0671__2.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodePr

Processing acoustic guitar:   2%|▏         | 12/637 [00:00<00:19, 31.59it/s]

Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\045__[gac][nod][jaz_blu]0823__1.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\045__[gac][nod][jaz_blu]0823__2.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\045__[gac][nod][jaz_blu]0823__3.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\046__[gac][dru][cou_fol]0709__1.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\acoustic guitar\046__[gac][dru][cou_fol]0709__2.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodePr

Processing acoustic guitar:   3%|▎         | 21/637 [00:00<00:19, 32.16it/s]

...

Processing voice:  99%|█████████▉| 771/778 [00:19<00:00, 41.35it/s]

Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2545__1.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2545__2.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2545__3.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2546__1.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2546__2.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2546__3.wav: cannot load libra

Processing voice: 100%|██████████| 778/778 [00:19<00:00, 39.09it/s]


Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2548__1.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2548__2.wav: cannot load library 'libsndfile.dll': error 0x7e
Error processing C:\Users\willi\VsCodeProjects\GitRepos\klink\data\raw\IRMAS-TrainingData\voice\[voi][pop_roc]2548__3.wav: cannot load library 'libsndfile.dll': error 0x7e


KeyError: "None of [Index(['file_name', 'label', 'spectral_centroid', 'spectral_bandwidth',\n       'spectral_rolloff', 'chroma_stft_mean', 'rms_mean', 'zcr', 'hnr_mean',\n       'onset_strength'],\n      dtype='str')] are in the [columns]"